# Audio/Video AI Pipeline: Transcribe → Translate → Redact → Sentiment → Serve

**Pipeline:** AI_TRANSCRIBE → AI_TRANSLATE → AI_REDACT → AI_SENTIMENT → Serve  
**Data:** 5 pre-recorded multi-speaker government service call recordings  
**Snowflake Features:** `AI_TRANSCRIBE`, `AI_TRANSLATE`, `AI_REDACT`, `AI_SENTIMENT`, `CORTEX.COMPLETE`

In [ ]:
USE DATABASE CDSB_DEMO;
USE SCHEMA RAW;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
!pip install plotly --quiet

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print('Connected to', session.get_current_database(), session.get_current_schema())

## 1. Pre-Staged Audio Recordings

Five multi-speaker WAV recordings of government service calls are pre-staged at `@AUDIO_FILES`.  
Each call features distinct caller and agent voices with realistic QLD scenarios:

| File | Scenario | Language | Duration |
|------|----------|----------|----------|
| `call_001_licence_complaint.wav` | Frustrated caller — delayed licence renewal | English | ~2.5 min |
| `call_002_registration_thankyou.wav` | Happy caller — smooth registration transfer | English | ~1.5 min |
| `call_003_elderly_portal_help.wav` | Confused elderly caller — online portal help | English | ~3 min |
| `call_004_bilingual_mandarin.wav` | Bilingual caller — Mandarin/English vehicle transfer | Mandarin | ~2 min |
| `call_005_fraud_report.wav` | Urgent fraud report — identity theft on registration | English | ~3 min |

In [ ]:
SELECT RELATIVE_PATH, SIZE, LAST_MODIFIED
FROM DIRECTORY(@AUDIO_FILES)
ORDER BY RELATIVE_PATH

In [ ]:
CREATE OR REPLACE TABLE CALL_SCENARIOS (
    CALL_ID NUMBER,
    FILENAME VARCHAR,
    SCENARIO VARCHAR,
    LANGUAGE VARCHAR,
    EXPECTED_SENTIMENT VARCHAR,
    CALLER_NAME VARCHAR,
    AGENT_NAME VARCHAR,
    REGION VARCHAR
);

INSERT INTO CALL_SCENARIOS VALUES
    (1, 'call_001_licence_complaint.wav', 'Frustrated caller about delayed licence renewal', 'en', 'negative', 'James Wilson', 'Sarah Chen', 'Brisbane'),
    (2, 'call_002_registration_thankyou.wav', 'Happy caller thanking agent for quick registration', 'en', 'positive', 'Emily Brown', 'David Kumar', 'Gold Coast'),
    (3, 'call_003_elderly_portal_help.wav', 'Confused elderly caller needing help with online portal', 'en', 'mixed', 'Margaret O''Brien', 'Lisa Nguyen', 'Toowoomba'),
    (4, 'call_004_bilingual_mandarin.wav', 'Bilingual caller needing vehicle transfer assistance', 'zh', 'mixed', 'Wei Zhang', 'Lucy Wang', 'Brisbane'),
    (5, 'call_005_fraud_report.wav', 'Caller reporting potential fraud on their account', 'en', 'negative', 'Sandra Martinez', 'Ben O''Connor', 'Mackay');

## 1b. AI_TRANSCRIBE — Speech to Text with Speaker Diarisation

Transcribe each call recording with **speaker labels** and **timestamps**.  
AI_TRANSCRIBE automatically detects the language and identifies distinct speakers.

**Supported formats:** FLAC, MP3, MP4, OGG, WAV, WEBM (audio) | MKV, MP4, OGV, WEBM (video)  
**Max duration:** 120 min (basic) / 60 min (with timestamps)  
**Cost:** ~$0.70 per hour of audio

In [ ]:
-- Preview: transcribe a single call with speaker identification
SELECT AI_TRANSCRIBE(
    TO_FILE('@AUDIO_FILES', 'call_001_licence_complaint.wav'),
    {'timestamp_granularity': 'speaker'}
) as TRANSCRIPTION

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE CALL_TRANSCRIPTS AS
    SELECT
        s.CALL_ID,
        s.FILENAME,
        s.SCENARIO,
        AI_TRANSCRIBE(
            TO_FILE('@AUDIO_FILES', s.FILENAME),
            {'timestamp_granularity': 'speaker'}
        ) as RAW_TRANSCRIPTION,
        TO_VARCHAR(AI_TRANSCRIBE(
            TO_FILE('@AUDIO_FILES', s.FILENAME)
        ):text) as TRANSCRIPT
    FROM CALL_SCENARIOS s
    ORDER BY s.CALL_ID
""").collect()

df_transcripts = session.sql("""
    SELECT CALL_ID, FILENAME, SCENARIO,
           RAW_TRANSCRIPTION:audio_duration::NUMBER(10,1) as DURATION_SECS,
           ARRAY_SIZE(RAW_TRANSCRIPTION:segments) as N_SEGMENTS,
           LEN(TRANSCRIPT) as TRANSCRIPT_LEN
    FROM CALL_TRANSCRIPTS
""").to_pandas()

print('Transcription Results:')
print(df_transcripts.to_string(index=False))
print(f'\nTotal audio duration: {df_transcripts["DURATION_SECS"].sum():.0f} seconds')

In [ ]:
df_turns = session.sql("""
    SELECT
        t.CALL_ID,
        s.value:speaker_label::VARCHAR as SPEAKER,
        ROUND(s.value:start::FLOAT, 1) as START_SEC,
        ROUND(s.value:end::FLOAT, 1) as END_SEC,
        LEFT(s.value:text::VARCHAR, 100) as TEXT_PREVIEW
    FROM CALL_TRANSCRIPTS t,
         LATERAL FLATTEN(input => RAW_TRANSCRIPTION:segments) s
    WHERE t.CALL_ID = 1
    ORDER BY START_SEC
""").to_pandas()

print('Speaker Turns — Call 001 (Licence Complaint):')
print(df_turns.to_string(index=False))
print(f'\nSpeakers detected: {df_turns["SPEAKER"].nunique()}')

## 2. AI_TRANSLATE — Multi-Language Translation

Translate call transcripts to support QLD's diverse population.  
Supports 24+ languages with auto-detection.

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE CALL_TRANSLATIONS AS
    SELECT
        t.CALL_ID,
        s.SCENARIO,
        s.LANGUAGE as ORIGINAL_LANGUAGE,
        t.TRANSCRIPT as ORIGINAL_TRANSCRIPT,
        AI_TRANSLATE(t.TRANSCRIPT, '', 'en') as ENGLISH_TRANSLATION,
        AI_TRANSLATE(t.TRANSCRIPT, '', 'zh') as CHINESE_TRANSLATION,
        AI_TRANSLATE(t.TRANSCRIPT, '', 'es') as SPANISH_TRANSLATION
    FROM CALL_TRANSCRIPTS t
    JOIN CALL_SCENARIOS s ON t.CALL_ID = s.CALL_ID
""").collect()

df_translations = session.sql('SELECT CALL_ID, SCENARIO, ORIGINAL_LANGUAGE, LEN(ENGLISH_TRANSLATION) as EN_LEN, LEN(CHINESE_TRANSLATION) as ZH_LEN, LEN(SPANISH_TRANSLATION) as ES_LEN FROM CALL_TRANSLATIONS').to_pandas()
print('Translation Results:')
df_translations

In [ ]:
SELECT
    CALL_ID,
    LEFT(ORIGINAL_TRANSCRIPT, 200) as ORIGINAL_PREVIEW,
    LEFT(SPANISH_TRANSLATION, 200) as SPANISH_PREVIEW
FROM CALL_TRANSLATIONS
WHERE CALL_ID = 1
LIMIT 1

## 3. AI_REDACT — PII Masking

Automatically detect and redact personally identifiable information from transcripts.  
Essential for compliance with QLD privacy legislation and federal Privacy Act.

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE CALL_REDACTED AS
    SELECT
        t.CALL_ID,
        s.SCENARIO,
        s.CALLER_NAME,
        t.ENGLISH_TRANSLATION as ORIGINAL_TEXT,
        AI_REDACT(t.ENGLISH_TRANSLATION) as REDACTED_TEXT,
        AI_REDACT(t.ENGLISH_TRANSLATION, NULL, NULL, 'detect') as PII_DETECTED
    FROM CALL_TRANSLATIONS t
    JOIN CALL_SCENARIOS s ON t.CALL_ID = s.CALL_ID
""").collect()

df_redacted = session.sql("""
    SELECT CALL_ID, SCENARIO, CALLER_NAME,
           LEFT(ORIGINAL_TEXT, 150) as ORIGINAL_PREVIEW,
           LEFT(REDACTED_TEXT, 150) as REDACTED_PREVIEW,
           ARRAY_SIZE(PII_DETECTED:spans) as PII_COUNT
    FROM CALL_REDACTED
""").to_pandas()

print('PII Redaction Results:')
print(f'Total PII instances detected: {df_redacted["PII_COUNT"].sum()}')
df_redacted[['CALL_ID','SCENARIO','PII_COUNT']]

In [ ]:
df_detail = session.sql("""
    SELECT
        CALL_ID,
        f.value:category::VARCHAR as PII_CATEGORY,
        f.value:text::VARCHAR as PII_TEXT
    FROM CALL_REDACTED,
         LATERAL FLATTEN(input => PII_DETECTED:spans) f
    ORDER BY CALL_ID, PII_CATEGORY
""").to_pandas()

print('Detected PII by Category:')
pii_summary = df_detail.groupby('PII_CATEGORY').size().sort_values(ascending=False)
print(pii_summary.to_string())

fig = px.bar(x=pii_summary.index, y=pii_summary.values,
             title='PII Categories Detected Across All Calls',
             labels={'x': 'PII Category', 'y': 'Count'},
             color=pii_summary.values, color_continuous_scale='Reds')
fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
SELECT
    CALL_ID,
    LEFT(ORIGINAL_TEXT, 500) as BEFORE_REDACTION,
    LEFT(REDACTED_TEXT, 500) as AFTER_REDACTION
FROM CALL_REDACTED
WHERE CALL_ID = 1

## 4. AI_SENTIMENT — Multi-Dimensional Sentiment Analysis

Analyse each call across multiple dimensions relevant to government service delivery.

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE CALL_SENTIMENT AS
    SELECT
        r.CALL_ID,
        s.SCENARIO,
        s.EXPECTED_SENTIMENT,
        s.REGION,
        AI_SENTIMENT(
            r.REDACTED_TEXT,
            ['Customer Satisfaction', 'Agent Professionalism', 'Issue Resolution',
             'Wait Time', 'Process Clarity', 'Empathy']
        ) as SENTIMENT_RESULT
    FROM CALL_REDACTED r
    JOIN CALL_SCENARIOS s ON r.CALL_ID = s.CALL_ID
""").collect()

df_sentiment = session.sql("""
    SELECT
        CALL_ID,
        SCENARIO,
        EXPECTED_SENTIMENT,
        REGION,
        f.value:name::VARCHAR as CATEGORY,
        f.value:sentiment::VARCHAR as SENTIMENT
    FROM CALL_SENTIMENT,
         LATERAL FLATTEN(input => SENTIMENT_RESULT:categories) f
    ORDER BY CALL_ID, CATEGORY
""").to_pandas()

overall = df_sentiment[df_sentiment['CATEGORY'] == 'overall'][['CALL_ID','SCENARIO','EXPECTED_SENTIMENT','SENTIMENT']]
overall.columns = ['CALL_ID','SCENARIO','EXPECTED','DETECTED']
print('Overall Sentiment Detection vs Expected:')
print(overall.to_string(index=False))

accuracy = (overall['EXPECTED'] == overall['DETECTED']).mean()
print(f'\nSentiment detection accuracy: {100*accuracy:.0f}%')

In [ ]:
categories = df_sentiment[df_sentiment['CATEGORY'] != 'overall'].copy()
sentiment_map = {'positive': 1, 'neutral': 0, 'negative': -1, 'mixed': 0.5, 'unknown': None}
categories['SCORE'] = categories['SENTIMENT'].map(sentiment_map)

pivot = categories.pivot_table(index='CALL_ID', columns='CATEGORY', values='SCORE', aggfunc='first')

fig = px.imshow(pivot, text_auto=False,
                color_continuous_scale='RdYlGn', zmin=-1, zmax=1,
                title='Call Sentiment Heatmap (green=positive, red=negative)',
                labels={'color': 'Sentiment Score'})
fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
region_sentiment = df_sentiment[df_sentiment['CATEGORY'] == 'overall'].copy()

fig = px.histogram(region_sentiment, x='REGION', color='SENTIMENT',
                   title='Call Sentiment by Region',
                   color_discrete_map={'positive': '#339D37', 'negative': '#E22339',
                                      'neutral': '#78797E', 'mixed': '#FFCC2C'},
                   barmode='stack')
fig.update_layout(template='plotly_white')
fig.show()

## 5. Serve — Final Analytics Table

Combine all pipeline outputs into a single analytics-ready table  
for dashboards, reporting, and Cortex Search.

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE CALL_ANALYTICS AS
    SELECT
        s.CALL_ID,
        s.SCENARIO,
        s.CALLER_NAME,
        s.AGENT_NAME,
        s.REGION,
        s.LANGUAGE as ORIGINAL_LANGUAGE,
        t.ORIGINAL_TRANSCRIPT,
        t.ENGLISH_TRANSLATION,
        r.REDACTED_TEXT,
        ARRAY_SIZE(r.PII_DETECTED:spans) as PII_COUNT,
        sent.SENTIMENT_RESULT:categories[0]:sentiment::VARCHAR as OVERALL_SENTIMENT,
        sent.SENTIMENT_RESULT as FULL_SENTIMENT,
        CURRENT_TIMESTAMP() as PROCESSED_AT
    FROM CALL_SCENARIOS s
    JOIN CALL_TRANSLATIONS t ON s.CALL_ID = t.CALL_ID
    JOIN CALL_REDACTED r ON s.CALL_ID = r.CALL_ID
    JOIN CALL_SENTIMENT sent ON s.CALL_ID = sent.CALL_ID
""").collect()

df_final = session.sql('SELECT CALL_ID, SCENARIO, CALLER_NAME, AGENT_NAME, REGION, ORIGINAL_LANGUAGE, PII_COUNT, OVERALL_SENTIMENT FROM CALL_ANALYTICS').to_pandas()
print('Final Analytics Table: CALL_ANALYTICS')
print(f'Columns: {session.sql("SELECT * FROM CALL_ANALYTICS LIMIT 0").to_pandas().columns.tolist()}')
df_final

## 6. LLM-Powered Call Centre Insights

Use Cortex AI to generate executive-level insights from the processed calls

In [ ]:
summary_data = session.sql("""
    SELECT
        REGION,
        OVERALL_SENTIMENT,
        PII_COUNT,
        SCENARIO
    FROM CALL_ANALYTICS
""").to_pandas().to_string(index=False)

llm_result = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'claude-sonnet-4-6',
        $$You are the Chief Customer Officer for the QLD Department of Transport and Main Roads.

Analyse these processed call centre records:

{summary_data}

Provide:
1. EXECUTIVE SUMMARY — key patterns in 3 bullet points
2. REGIONAL ANALYSIS — which regions need attention
3. AGENT PERFORMANCE — any concerns from the call data
4. PII COMPLIANCE — assessment of data handling
5. RECOMMENDATIONS — top 3 actions for next quarter
6. PIPELINE VALUE — how this automated pipeline benefits the department$$
    ) as INSIGHTS
""").to_pandas()

print(llm_result.iloc[0]['INSIGHTS'])

## Pipeline Summary

```
┌─────────────┐     ┌──────────────┐     ┌────────────┐     ┌──────────────┐     ┌─────────┐
│ Audio/Video  │────▶│ AI_TRANSCRIBE│────▶│AI_TRANSLATE│────▶│  AI_REDACT   │────▶│  Serve  │
│   Files      │     │  Speech→Text │     │ Multi-lang │     │  PII Mask    │     │Analytics│
└─────────────┘     └──────────────┘     └────────────┘     └──────────────┘     └─────────┘
                                                                    │
                                                            ┌──────────────┐
                                                            │ AI_SENTIMENT │
                                                            │ Multi-dim    │
                                                            └──────────────┘
```

| Step | Function | Input | Output | Benefit |
|------|----------|-------|--------|--------|
| 1 | `AI_TRANSCRIBE` | Audio/Video files | Text + speaker labels | Digitise call recordings |
| 2 | `AI_TRANSLATE` | Any language text | English + target langs | Support diverse population |
| 3 | `AI_REDACT` | Raw transcript | PII-safe text | Privacy compliance |
| 4 | `AI_SENTIMENT` | Redacted text | Multi-dim sentiment | Quality monitoring |
| 5 | Serve | All outputs | Analytics table | Dashboard-ready data |

**All processing happens inside Snowflake — no data movement, no external APIs, fully governed.**